# Paper figures for the revision

This notebook makes every figure proposed for the revised manuscript. Each figure is built from three sources:

- the analysis tables, which give the bootstrap CIs and summaries
- the saved per-image predictions of every run, which give the confusion matrices, ROC curves and calibration
- the training logs of every run (`train_result.json`), which give the training curves (F20–F22)

The predictions are read through the analysis script's own loader (`build_revision_analysis.discover` / `load_all`). So every figure uses exactly the runs, labels and image IDs that the tables use. Section 2 checks this.

**It runs in two places.** The setup cell detects which:

| | Results package (`D:\Afterimage_data_share\afterimage_results_<date>\`) | Repository |
|---|---|---|
| Run records | `experiments/` in the package | `D:\Afterimage_Experiments` |
| Tables | **rebuilt from the run records** into `rebuilt_tables/`, then checked against `tables/` | `finetuning/paper_tables/output/revision_analysis/` |
| Figures | `figures/` | `finetuning/paper_figures/output/` |
| Last cell | skipped | zips the report bundle for ChatGPT |

**How to run.**

- **Kernel:** `dr_paper_repro` (or any Python 3.11 with numpy, pandas, scipy and matplotlib). `afterimage_env` has no matplotlib.
- **Run:** use *Run All*. No GPU work. In the package, rebuilding the tables takes a few minutes; the figures take about a minute.
- **Layout check:** every figure is checked for overlapping text before it is saved, and the result is printed under it. The last section lists any figure that failed and writes `run_report.txt`.

**Output** goes to the figures folder:

- `Fxx_name.png`, at 300 dpi
- `Fxx_name.pdf`, vector, for the journal
- `Fxx_name_data.csv`, the exact numbers plotted
- `figure_index.csv`, which lists every figure with its data sources and a draft caption

| ID | Figure | Placement |
|---|---|---|
| F01 | Cohorts: grade mix of APTOS splits, archived Messidor and verified Messidor-2; class balancing in training | Main |
| F02 | Headline performance, in-domain vs external, every seed | Main |
| F03 | Forest plot of every paired comparison (Δ accuracy, Δ QWK, 95% CI) | Main |
| F04 | LoRA rank and alpha grid at lr 2e-4 against lr 5e-4 | Supplementary |
| F05 | Learning rate chosen on validation loss; failed runs separated by validation loss | Supplementary |
| F06 | With and without the LoRA adapter | Main or supplementary |
| F07 | CLAHE on vs off, both models, both cohorts | Supplementary |
| F08 | Confusion matrices, Qwen vs RetiZero, APTOS vs Messidor-2 | Main |
| F09 | Error distance: exact, ±1, ±2, ≥3 grades | Supplementary |
| F10 | Class-wise sensitivity and specificity | Supplementary |
| F11 | One-vs-rest ROC curves per grade | Supplementary |
| F12 | Referable-DR ROC with each model's operating point | Main |
| F13 | Mild vs Normal: the main failure on Messidor | Supplementary |
| F14 | Predicted vs true grade distribution (under-grading) | Supplementary |
| F15 | Calibration: reliability diagrams and ECE | Supplementary |
| F16 | Efficiency: parameters, training time, inference time, memory, energy | Main |
| F17 | Seed-to-seed variability of every APTOS condition | Supplementary |
| F18 | Near-duplicate sensitivity | Supplementary |
| F19 | Archived vs verified Messidor agreement | Supplementary |
| F20 | Qwen training and validation loss by learning rate (from the training logs) | Supplementary |
| F21 | RetiZero training curves (from the training logs) | Supplementary |
| F22 | Validation loss across the LoRA rank and alpha grid at lr 2e-4 (from the training logs) | Supplementary |
| F23 | Study workflow (methodology diagram) | Main |
| F24 | Qwen3-VL-8B architecture with the LoRA adapter and five-answer scoring | Main |
| F25 | RetiZero architecture with the new classification head | Main |

**Not generated here.** Grad-CAM and misclassified-image examples need the fundus images, which stay out of the repository. The contact sheets in `image_audits\` outside the repo can serve as the misclassification figure.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import re
import json
import platform
from datetime import datetime
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Where are we? Jupyter and VS Code start a notebook in its own folder.
HERE = Path.cwd()
PACKAGE = (HERE / "experiments").is_dir() and (HERE / "code" / "paper_tables" / "build_revision_analysis.py").is_file()
if PACKAGE:
    ROOT = HERE
    EXPERIMENTS = ROOT / "experiments"
    CODE = ROOT / "code" / "paper_tables"
    INPUTS = ROOT / "inputs"
    SHIPPED_TABLES = ROOT / "tables"
    REBUILD_TABLES = True            # False: draw from tables/ without rebuilding
    TABLES = ROOT / "rebuilt_tables" if REBUILD_TABLES else SHIPPED_TABLES
    OUT = ROOT / "figures"
else:
    REPO = Path(r"D:\Afterimage_Reproduction")
    EXPERIMENTS = Path(r"D:\Afterimage_Experiments")
    CODE = REPO / "finetuning" / "paper_tables"
    REBUILD_TABLES = False           # in the repo, build the tables with build_revision_analysis.py
    TABLES = CODE / "output" / "revision_analysis"
    OUT = REPO / "finetuning" / "paper_figures" / "output"
assert EXPERIMENTS.is_dir(), f"Run records not found: {EXPERIMENTS}"
OUT.mkdir(parents=True, exist_ok=True)

sys.dont_write_bytecode = True
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import build_revision_analysis as bra
assert Path(bra.__file__).resolve().parent == CODE.resolve(), f"Imported the analysis code from the wrong place: {bra.__file__}"
REPORT = [f"# Figure notebook run report", "", f"- Started {datetime.now():%Y-%m-%d %H:%M:%S}",
          f"- Mode: {'results package' if PACKAGE else 'repository'}", f"- Run records: `{EXPERIMENTS}`",
          f"- Tables: `{TABLES}`" + (" (rebuilt here)" if REBUILD_TABLES else ""), f"- Figures: `{OUT}`",
          f"- Python {platform.python_version()}, numpy {np.__version__}, pandas {pd.__version__}, "
          f"scipy {scipy.__version__}, matplotlib {matplotlib.__version__}"]

# Constrained layout places titles, labels, tick labels and the legends outside the axes so they do not collide.
plt.rcParams.update({
    "figure.constrained_layout.use": True, "figure.constrained_layout.h_pad": 0.06, "figure.constrained_layout.w_pad": 0.06,
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "grid.linewidth": 0.5, "axes.axisbelow": True, "figure.dpi": 110, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.05, "pdf.fonttype": 42,
})

# Condition names as the analysis writes them, with short labels and one colour each (Okabe-Ito, colour-blind safe).
Q_ON_2E4, Q_OFF_2E4 = "Qwen r16 CLAHE-on lr2e-4", "Qwen r16 CLAHE-off lr2e-4"
Q_ON_5E4, Q_OFF_5E4 = "Qwen r16 CLAHE-on lr5e-4", "Qwen r16 CLAHE-off lr5e-4"
Z_ON, Z_OFF = "Qwen zero-shot 4bit CLAHE-on", "Qwen zero-shot 4bit CLAHE-off"
R_OFF, R_ON, R_APP = "RetiZero CLAHE-off", "RetiZero CLAHE-on", "RetiZero saved app checkpoint"
STYLE = {
    Q_ON_2E4: ("Qwen LoRA, lr 2e-4, CLAHE on (headline)", "#0072B2"),
    Q_OFF_2E4: ("Qwen LoRA, lr 2e-4, CLAHE off", "#56B4E9"),
    Q_ON_5E4: ("Qwen LoRA, lr 5e-4 (original), CLAHE on", "#CC79A7"),
    Q_OFF_5E4: ("Qwen LoRA, lr 5e-4, CLAHE off", "#E7B8D4"),
    Z_ON: ("Qwen zero-shot, CLAHE on", "#7F7F7F"),
    Z_OFF: ("Qwen zero-shot, CLAHE off", "#BDBDBD"),
    R_OFF: ("RetiZero, CLAHE off (native)", "#E69F00"),
    R_ON: ("RetiZero, CLAHE on", "#D55E00"),
    R_APP: ("RetiZero saved app checkpoint", "#C9B200"),
}
LR_COLOUR = {"1e-4": "#009E73", "2e-4": "#0072B2", "5e-4": "#CC79A7"}
COHORT = {"aptos361": "APTOS-361 (in-domain)", "messidor": "Messidor, archived (1,057)",
          "messidor2": "Messidor-2, verified (1,744)"}
SHORT_COHORT = {"aptos361": "APTOS-361", "messidor": "Messidor\narchived", "messidor2": "Messidor-2\nverified"}
GRADE = ["0 No DR", "1 Mild", "2 Moderate", "3 Severe", "4 PDR"]
GRADE_COLOURS = plt.cm.YlOrRd(np.linspace(0.12, 0.92, 5))
FULL_WIDTH = 7.2   # inches, a double-column journal figure


def label(cond):
    return STYLE.get(cond, (cond, "#444444"))[0]


def colour(cond):
    return STYLE.get(cond, (cond, "#444444"))[1]


def legend_below(fig, handles, ncol):
    """One legend for the whole figure, under the panels, never on top of data or labels."""
    fig.legend(handles=handles, loc="outside lower center", ncol=ncol, handlelength=1.4, columnspacing=1.2)


def drawn_text_boxes(fig):
    """Bounding boxes of every piece of text that is actually drawn, plus legend boxes."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    items = []

    def add(t, kind):
        if t is not None and t.get_visible() and t.get_text().strip():
            bb = t.get_window_extent(r)
            if bb.width > 1 and bb.height > 1:
                items.append((kind, t.get_text().replace("\n", " ")[:45], bb))

    for ax in fig.axes:
        if not ax.get_visible():
            continue
        for t in (ax.title, ax._left_title, ax._right_title, ax.xaxis.label, ax.yaxis.label, *ax.texts):
            add(t, "text")
        for axis, (lo, hi) in ((ax.xaxis, sorted(ax.get_xlim())), (ax.yaxis, sorted(ax.get_ylim()))):
            for tick in axis.get_major_ticks():
                if lo - 1e-9 <= tick.get_loc() <= hi + 1e-9 and tick.label1.get_visible():
                    add(tick.label1, "tick")
        leg = ax.get_legend()
        if leg is not None:
            items.append(("legend", "axes legend", leg.get_window_extent(r)))
    for leg in fig.legends:
        items.append(("legend", "figure legend", leg.get_window_extent(r)))
    for t in fig.texts:
        add(t, "text")
    return items


def overlaps(fig, tolerance=1.0):
    """Pairs of drawn text (or legend) boxes that overlap by more than `tolerance` pixels."""
    items = drawn_text_boxes(fig)
    hits = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            (ka, ta, a), (kb, tb, b) = items[i], items[j]
            if "legend" in (ka, kb) and (ka == "tick" or kb == "tick"):
                continue
            w = min(a.x1, b.x1) - max(a.x0, b.x0)
            h = min(a.y1, b.y1) - max(a.y0, b.y0)
            if w > tolerance and h > tolerance:
                hits.append(f"'{ta}' overlaps '{tb}'")
    return hits


INDEX, LAYOUT_PROBLEMS = [], {}


def save(fig, fid, name, title, sources, caption, data=None):
    """Check the layout, write PNG + PDF (+ the plotted numbers as CSV) and record the figure in the index."""
    stem = f"{fid}_{name}"
    hits = overlaps(fig)
    if hits:
        LAYOUT_PROBLEMS[fid] = hits
    fig.savefig(OUT / f"{stem}.png")
    fig.savefig(OUT / f"{stem}.pdf")
    if data is not None:
        data.to_csv(OUT / f"{stem}_data.csv", index=False)
    INDEX.append({"id": fid, "file": f"{stem}.png", "pdf": f"{stem}.pdf",
                  "data": f"{stem}_data.csv" if data is not None else "", "title": title,
                  "sources": sources, "draft_caption": caption})
    plt.show()
    plt.close(fig)
    print(f"saved {stem}  |  layout check: " + ("no overlapping text" if not hits else f"{len(hits)} OVERLAPS: " + "; ".join(hits[:5])))


def bars_with_seeds(ax, x, values, width, col):
    """A bar at the mean with every seed as a black dot."""
    values = np.asarray(values, float)
    ax.bar(x, values.mean(), width, color=col)
    ax.scatter(np.full(len(values), x), values, s=7, color="black", zorder=3, linewidths=0)


print("\n".join(REPORT[2:]))

## 1b. Rebuild the analysis tables from the run records (results package only)

First every input file is checked against `SHA256SUMS.txt`. Then `build_revision_analysis.py` recomputes every table from the per-image predictions and training records in `experiments/`: metrics, bootstrap CIs (2,000 resamples, fixed seed), McNemar tests, calibration, efficiency and timing. Finally each rebuilt table is compared with the one in `tables/`. They must agree to 1e-9; the cell stops if any differs.

In [ ]:
import argparse
import hashlib
import time


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def same_table(a_path, b_path, tol=1e-9):
    a, b = pd.read_csv(a_path), pd.read_csv(b_path)
    if list(a.columns) != list(b.columns) or len(a) != len(b):
        return f"shape or columns differ: {a.shape} vs {b.shape}"
    worst = 0.0
    for col in a.columns:
        x, y = a[col], b[col]
        if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
            x, y = x.to_numpy(float), y.to_numpy(float)
            if not np.array_equal(np.isnan(x), np.isnan(y)):
                return f"{col}: missing values differ"
            d = np.nanmax(np.abs(x - y)) if len(x) and not np.isnan(x).all() else 0.0
            worst = max(worst, d)
            if d > tol:
                return f"{col}: differs by {d:.3g}"
        elif not x.fillna("").astype(str).equals(y.fillna("").astype(str)):
            return f"{col}: text differs"
    return None


if PACKAGE:
    sums = [line.split("  ", 1) for line in (ROOT / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines() if line.strip()]
    bad = [rel for digest, rel in sums if not (ROOT / rel).is_file() or file_sha256(ROOT / rel) != digest]
    assert not bad, f"{len(bad)} input files are missing or changed, e.g. {bad[:5]}"
    print(f"Checksums: all {len(sums):,} input files match SHA256SUMS.txt")
    REPORT.append(f"- Checksums: {len(sums):,} input files match `SHA256SUMS.txt`")

if REBUILD_TABLES:
    t0 = time.time()
    build_args = argparse.Namespace(
        experiments_root=EXPERIMENTS,
        messidor_grades=INPUTS / "messidor2" / "messidor_data.csv",
        messidor_pairs=INPUTS / "messidor2" / "pairs.csv",
        messidor_duplicates=INPUTS / "image_audits" / "messidor_duplicate_pairs.csv",
        original_split=EXPERIMENTS / "reference" / "aptos_split",
        near_duplicates=INPUTS / "image_audits" / "near_duplicate_test_ids.csv",
        out=TABLES)
    result = bra.build(build_args)
    print(f"Rebuilt {result['outputs']} tables from {result['runs']} runs in {time.time() - t0:.0f} s -> {TABLES}")
    differences = {}
    shipped = sorted(SHIPPED_TABLES.glob("*.csv"))
    for path in shipped:
        new = TABLES / path.name
        problem = "not rebuilt" if not new.is_file() else same_table(path, new)
        if problem:
            differences[path.name] = problem
    (TABLES / "SUMMARY.md").unlink(missing_ok=True)  # a text digest of the tables; the package ships the CSVs only
    REPORT.append(f"- Tables rebuilt from {result['runs']} runs in {time.time() - t0:.0f} s; "
                  f"{len(shipped) - len(differences)} of {len(shipped)} equal the shipped tables (1e-9)")
    assert not differences, f"Rebuilt tables differ from tables/: {differences}"
    print(f"All {len(shipped)} rebuilt tables equal the shipped tables.")
else:
    print("Tables are read from", TABLES)

## 2. Load the tables and the per-image predictions, and check that they agree

In [ ]:
per_run = pd.read_csv(TABLES / "per_run_metrics.csv")
runs = bra.discover(EXPERIMENTS)
bra.load_all(runs)
print(f"{len(runs)} runs with predictions; per_run_metrics.csv has {len(per_run)} rows")

# Every run in the table must be the run whose predictions we plot, with the same accuracy.
check = pd.DataFrame([{"job": r["name"], "cohort": r["cohort"], "acc_pred": r["metrics"]["accuracy"]} for r in runs])
merged = per_run.merge(check, on=["job", "cohort"], how="outer", indicator=True)
assert (merged["_merge"] == "both").all(), merged[merged["_merge"] != "both"][["job", "cohort", "_merge"]]
assert (merged["accuracy"] - merged["acc_pred"]).abs().max() < 1e-8   # the CSV stores rounded floats
print("Tables and predictions agree for every run.")


def runs_of(cond, cohort):
    return [r for r in runs if r["condition"] == cond and r["cohort"] == cohort]


def pooled(cond, cohort):
    """All seeds' predictions stacked (for confusion matrices and calibration, which pool seeds)."""
    return pd.concat([r["frame"].assign(seed=r["seed"]) for r in runs_of(cond, cohort)], ignore_index=True)


def metric(cond, cohort, m):
    return per_run[(per_run.condition == cond) & (per_run.cohort == cohort)][m].to_numpy(float)


def failed(n_grades):
    return n_grades <= 3   # the failure definition used throughout REVISION_EVIDENCE.md


def roc(y, s):
    order = np.argsort(-s, kind="mergesort")
    y, s = y[order], s[order]
    distinct = np.r_[np.where(np.diff(s))[0], len(s) - 1]
    tps = np.cumsum(y)[distinct]
    fps = np.cumsum(1 - y)[distinct]
    return np.r_[0, fps / max(fps[-1], 1)], np.r_[0, tps / max(tps[-1], 1)]

## F01. Cohorts and class balance

In [ ]:
counts = pd.read_csv(TABLES / "class_counts.csv")
clean = counts[counts.split_set.str.startswith("clean")]
disk = clean[clean.view == "images on disk (both models)"].set_index("split")
rows = [{"cohort": name, **{f"g{g}": float(disk.loc[split, f"g{g}"]) for g in range(5)}}
        for split, name in (("train", "APTOS train"), ("val", "APTOS validation"), ("test", "APTOS test"))]
for cohort, name in (("messidor", "Messidor archived"), ("messidor2", "Messidor-2 verified")):
    y = runs_of(Q_ON_2E4, cohort)[0]["frame"]["y_true"].value_counts().reindex(range(5), fill_value=0)
    rows.append({"cohort": name, **{f"g{g}": float(y[g]) for g in range(5)}})
f01 = pd.DataFrame(rows)
f01["n"] = f01[[f"g{g}" for g in range(5)]].sum(axis=1)

fig, (a, b) = plt.subplots(1, 2, figsize=(FULL_WIDTH, 3.0), width_ratios=[1.6, 1])
left = np.zeros(len(f01))
for g in range(5):
    share = (f01[f"g{g}"] / f01["n"]).to_numpy()
    a.barh(range(len(f01)), share, left=left, color=GRADE_COLOURS[g], edgecolor="white", linewidth=0.5)
    for i, (s, l) in enumerate(zip(share, left)):
        if s >= 0.09:
            a.text(l + s / 2, i, f"{s:.0%}", ha="center", va="center", fontsize=6, color="black" if g < 3 else "white")
    left += share
a.set_yticks(range(len(f01)), [f"{c} (n = {int(n):,})" for c, n in zip(f01.cohort, f01.n)])
a.invert_yaxis(); a.set_xlim(0, 1); a.grid(False)
a.set_xlabel("Share of images"); a.set_title("A  Grade mix of each cohort", loc="left")

train = clean[clean.split == "train"]
disk_row = train[train.view.str.startswith("images on disk")].iloc[0]
qwen_row = train[train.view.str.startswith("Qwen")].iloc[0]
x = np.arange(5)
b.bar(x - 0.2, [disk_row[f"g{g}"] for g in range(5)], 0.38, color="#999999")
b.bar(x + 0.2, [qwen_row[f"g{g}"] for g in range(5)], 0.38, color="#0072B2")
b.set_xticks(x, [str(g) for g in range(5)]); b.set_xlabel("Grade"); b.set_ylabel("Images per epoch")
b.set_title("B  Training images per epoch (APTOS train)", loc="left")
legend_below(fig, [Patch(color=GRADE_COLOURS[g], label=GRADE[g]) for g in range(5)]
             + [Patch(color="#999999", label="On disk (RetiZero sees these; loss weighted)"),
                Patch(color="#0072B2", label="Qwen draws (weighted sampler)")], ncol=4)
save(fig, "F01", "cohorts_and_class_balance", "Cohorts and class balance",
     "class_counts.csv; per-image labels of the Messidor runs",
     "(A) Grade distribution of the clean APTOS splits, the archived Messidor cohort and the verified Messidor-2 "
     "cohort (1,744 gradable images, adjudicated grades); shares below 9% are not labelled. (B) Training images per "
     "grade per epoch: Qwen draws a class-balanced sample with an inverse-frequency weighted sampler; RetiZero sees "
     "every image once, with an inverse-frequency weighted loss and online augmentation.", f01)

## F02. Headline performance, in-domain vs external

In [ ]:
conds = [Z_ON, Q_ON_5E4, Q_ON_2E4, R_OFF, R_ON]
cohorts = ["aptos361", "messidor2"]
metrics = [("accuracy", "Accuracy (%)", 100), ("qwk", "Quadratic weighted kappa", 1), ("macro_auc", "Macro AUC", 1)]
f02 = per_run[per_run.condition.isin(conds) & per_run.cohort.isin(cohorts)][["condition", "cohort", "seed", "accuracy", "qwk", "macro_auc"]]
fig, axes = plt.subplots(1, 3, figsize=(FULL_WIDTH, 3.0))
w = 0.16
for ax, (m, name, scale) in zip(axes, metrics):
    for ci, cohort in enumerate(cohorts):
        for k, cond in enumerate(conds):
            v = metric(cond, cohort, m) * scale
            if len(v):
                bars_with_seeds(ax, ci + (k - 2) * w, v, w * 0.9, colour(cond))
    ax.set_xticks([0, 1], ["APTOS-361", "Messidor-2"])
    ax.set_title(name)
    ax.set_ylim(0, 100 if m == "accuracy" else 1)
legend_below(fig, [Patch(color=colour(c), label=label(c)) for c in conds], ncol=3)
save(fig, "F02", "headline_performance", "Headline performance in-domain and on Messidor-2",
     "per_run_metrics.csv",
     "Accuracy, QWK and macro AUC on the 361 clean APTOS test images and on the 1,744 verified Messidor-2 images "
     "(trained on APTOS only). Bars: mean over three seeds (one run for zero-shot); dots: individual seeds.", f02)

## F03. Every paired comparison in one forest plot

Each row is Δ = A − B with a 95% hierarchical bootstrap CI. Filled marker: the interval excludes zero.

In [ ]:
files = ["head_to_head_aggregate.csv", "messidor_aggregate.csv", "messidor2_aggregate.csv", "training_aggregate.csv",
         "clahe_aggregate.csv", "rank_lr2e-4_aggregate.csv", "rank_aggregate.csv"]
f03 = pd.concat([pd.read_csv(TABLES / f) for f in files], ignore_index=True)
# the analysis script prefixes some comparisons with internal codes ("H1 proposed primary: ", "H2 ", "S7 "); drop them
f03["comparison"] = (f03["comparison"].str.replace(r"^H1 proposed primary: (.*)$", r"\1 (proposed primary)", regex=True)
                     .str.replace(r"^(H\d|S7) ", "", regex=True))
family_name = {"head_to_head": "Qwen vs RetiZero, APTOS-361", "messidor": "Qwen vs RetiZero, archived Messidor",
               "messidor2": "Verified Messidor-2", "learning_rate": "Learning rate (Qwen r16, CLAHE on)",
               "adapter": "With vs without the LoRA adapter", "clahe": "CLAHE on vs off, APTOS-361",
               "rank_lr2e-4": "LoRA rank at lr 2e-4", "rank": "LoRA rank at lr 5e-4 (unstable rate)"}


def short(text):
    text = text.replace("Qwen (lr chosen by validation loss) vs RetiZero, native preprocessing (proposed primary)",
                        "Qwen lr 2e-4 vs RetiZero, native (primary)")
    text = text.replace("Messidor-2: ", "").replace("Messidor: ", "")
    text = text.replace(", native preprocessing", ", native").replace(" with alpha = rank (alpha/r = 1)", ", alpha = rank")
    text = text.replace("Qwen original paper setting", "Qwen lr 5e-4 (original)").replace("Qwen LoRA adapter", "Adapter")
    return text


f03["group"] = f03["family"].map(family_name).fillna(f03["family"])
f03["label"] = f03["comparison"].map(short)
groups = list(dict.fromkeys(f03["group"]))
ypos, ticks, y = [], [], 0.0
for g in groups:
    ticks.append((y, g, True)); y += 1
    for _ in range(int((f03.group == g).sum())):
        ypos.append(y); ticks.append((y, None, False)); y += 1
    y += 0.5
f03["y"] = ypos
labels_by_y = dict(zip(f03.y, f03.label))
fig, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 0.17 * y + 0.9), sharey=True, width_ratios=[1, 1])
for ax, (m, scale, name) in zip(axes, [("accuracy", 100, "Δ accuracy (percentage points)"), ("qwk", 1, "Δ QWK")]):
    lo, hi, mid = f03[f"delta_{m}_ci_low"] * scale, f03[f"delta_{m}_ci_high"] * scale, f03[f"delta_{m}"] * scale
    sig = (lo > 0) | (hi < 0)
    ax.hlines(f03.y, lo, hi, color="#555555", linewidth=0.8)
    ax.scatter(mid, f03.y, s=14, c=np.where(sig, "#0072B2", "white"), edgecolors="#0072B2", zorder=3)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_xlabel(name)
axes[0].set_yticks([p for p, _, _ in ticks], [g if head else labels_by_y[p] for p, g, head in ticks])
for tick, (_, _, head) in zip(axes[0].get_yticklabels(), ticks):
    tick.set_fontweight("bold" if head else "normal")
    tick.set_fontsize(6.5)
axes[0].invert_yaxis()
legend_below(fig, [Line2D([], [], marker="o", ls="", mfc="#0072B2", mec="#0072B2", label="95% CI excludes 0"),
                   Line2D([], [], marker="o", ls="", mfc="white", mec="#0072B2", label="95% CI includes 0")], ncol=2)
save(fig, "F03", "paired_comparisons_forest", "All paired comparisons", ", ".join(files),
     "Paired differences (A − B) with 95% hierarchical bootstrap confidence intervals (images × seeds; exams × seeds on "
     "Messidor-2; patients × seeds on archived Messidor). Filled markers: interval excludes zero.", f03.drop(columns=["y"]))

## F04. LoRA rank and alpha at the selected learning rate

In [ ]:
configs = [(8, 16), (8, 8), (16, 16), (32, 16), (32, 32)]
rows = []
for clahe in ("on", "off"):
    for rank, alpha in configs:
        for lr in ("2e-4", "5e-4"):
            cond = f"Qwen r{rank} CLAHE-{clahe} lr{lr}" + (f" alpha{alpha}" if alpha != 16 else "")
            for _, r in per_run[(per_run.condition == cond) & (per_run.cohort == "aptos361")].iterrows():
                rows.append({"clahe": clahe, "rank": rank, "alpha": alpha, "lr": lr, "seed": r.seed, "condition": cond,
                             "accuracy": r.accuracy, "qwk": r.qwk, "n_grades_predicted": r.n_grades_predicted})
f04 = pd.DataFrame(rows)
fig, axes = plt.subplots(2, 2, figsize=(FULL_WIDTH, 4.6), sharex=True)
for i, clahe in enumerate(("on", "off")):
    for j, (m, scale, name) in enumerate([("accuracy", 100, "Accuracy (%)"), ("qwk", 1, "QWK")]):
        ax = axes[i, j]
        for k, (rank, alpha) in enumerate(configs):
            for lr, dx, col in (("2e-4", -0.14, "#0072B2"), ("5e-4", 0.14, "#AAAAAA")):
                d = f04[(f04.clahe == clahe) & (f04["rank"] == rank) & (f04.alpha == alpha) & (f04.lr == lr)]
                if d.empty:
                    continue
                ok, bad = d[~failed(d.n_grades_predicted)], d[failed(d.n_grades_predicted)]
                ax.scatter(np.full(len(ok), k + dx), ok[m] * scale, s=12, color=col, zorder=3)
                ax.scatter(np.full(len(bad), k + dx), bad[m] * scale, s=22, marker="x", color="#D55E00", zorder=3)
                ax.hlines(d[m].mean() * scale, k + dx - 0.1, k + dx + 0.1, color=col, linewidth=1.5)
        ax.set_ylabel(name)
        ax.set_title(f"{'Accuracy' if m == 'accuracy' else 'QWK'}, CLAHE {clahe}", loc="left")
        ax.set_ylim((0, 100) if m == "accuracy" else (-0.05, 1))
        ax.set_xticks(range(len(configs)), [f"rank {r}\nα {a}" for r, a in configs])
legend_below(fig, [Line2D([], [], marker="o", ls="", color="#0072B2", label="lr 2e-4 (selected; mean = line)"),
                   Line2D([], [], marker="o", ls="", color="#AAAAAA", label="lr 5e-4 (original)"),
                   Line2D([], [], marker="x", ls="", color="#D55E00", label="failed run (≤ 3 grades predicted)")], ncol=3)
save(fig, "F04", "lora_rank_alpha_grid", "LoRA rank and alpha grid",
     "per_run_metrics.csv (rank_lr2e-4_aggregate.csv for the CIs)",
     "APTOS-361 accuracy and QWK for LoRA rank 8, 16 and 32 with alpha 16, and with alpha = rank, CLAHE on and off. At the "
     "validation-selected learning rate (2e-4) every run predicts all five grades and no rank-vs-16 difference is "
     "significant; at the original 5e-4 several runs fail (no 5e-4 runs exist for alpha = rank at ranks 8 and 32). "
     "Dots: seeds; lines: means.", f04)

## F05. Learning rate chosen on validation data

In [ ]:
cv = pd.read_csv(TABLES / "collapse_vs_validation.csv")
cv["lr"] = cv.condition.str.extract(r"lr(\de-\d)")[0]
cv["failed"] = failed(cv.n_grades_predicted)
lrsel = pd.read_csv(TABLES / "lr_selection_by_validation_loss.csv")
lrsel["lr"] = lrsel.condition.str.extract(r"lr(\de-\d)")[0]
fig, (a, b) = plt.subplots(1, 2, figsize=(FULL_WIDTH, 3.0), width_ratios=[1.5, 1])
gap = (cv[~cv.failed].val_loss.max(), cv[cv.failed].val_loss.min())
a.axvspan(*gap, color="#DDDDDD", alpha=0.7, lw=0)
for lr, d in cv.groupby("lr"):
    a.scatter(d[~d.failed].val_loss, d[~d.failed].accuracy * 100, s=14, color=LR_COLOUR[lr])
    a.scatter(d[d.failed].val_loss, d[d.failed].accuracy * 100, s=26, marker="x", color=LR_COLOUR[lr])
a.set_xlabel("Best validation loss"); a.set_ylabel("APTOS-361 accuracy (%)"); a.set_ylim(0, 100)
a.set_title(f"A  All {len(cv)} fine-tuned Qwen runs", loc="left")
x = np.arange(len(lrsel))
b.bar(x, lrsel.val_loss_mean, yerr=lrsel.val_loss_sd.fillna(0), color=[LR_COLOUR[l] for l in lrsel.lr], capsize=2)
b.set_xticks(x, [f"lr {l}\ntest {100 * t:.1f}%" for l, t in zip(lrsel.lr, lrsel.test_accuracy_mean)])
b.set_ylabel("Mean validation loss"); b.set_ylim(0, (lrsel.val_loss_mean + lrsel.val_loss_sd.fillna(0)).max() * 1.1)
b.set_title("B  Rank 16, CLAHE on", loc="left")
legend_below(fig, [Line2D([], [], marker="o", ls="", color=c, label=f"lr {k}") for k, c in LR_COLOUR.items()]
             + [Line2D([], [], marker="x", ls="", color="black", label="failed run (≤ 3 grades)"),
                Patch(color="#DDDDDD", label="gap between working and failed runs")], ncol=5)
save(fig, "F05", "learning_rate_validation_loss", "Learning rate selection and failed runs",
     "collapse_vs_validation.csv, lr_selection_by_validation_loss.csv",
     "(A) Validation loss against APTOS test accuracy for every fine-tuned Qwen run; failed runs (at most three grades "
     "predicted) have validation losses that do not overlap with working runs (grey band). (B) Mean validation loss by "
     "learning rate (rank 16, CLAHE on, three seeds; test accuracy under each bar); the lowest validation loss selects "
     "2e-4 without using the test set.", cv)

## F06. With and without the LoRA adapter

In [ ]:
groups6 = [("aptos361", "on", [Z_ON, Q_ON_5E4, Q_ON_2E4]), ("aptos361", "off", [Z_OFF, Q_OFF_5E4, Q_OFF_2E4]),
           ("messidor2", "on", [Z_ON, Q_ON_5E4, Q_ON_2E4]), ("messidor2", "off", [Z_OFF, Q_OFF_5E4, Q_OFF_2E4])]
rows = []
for cohort, clahe, cs in groups6:
    for cond in cs:
        for v_acc, v_qwk in zip(metric(cond, cohort, "accuracy"), metric(cond, cohort, "qwk")):
            rows.append({"cohort": cohort, "clahe": clahe, "condition": cond, "accuracy": v_acc, "qwk": v_qwk})
f06 = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 2.9))
for ax, (m, scale, name) in zip(axes, [("accuracy", 100, "Accuracy (%)"), ("qwk", 1, "QWK")]):
    for gi, (cohort, clahe, cs) in enumerate(groups6):
        for k, cond in enumerate(cs):
            v = f06[(f06.cohort == cohort) & (f06.condition == cond)][m] * scale
            if len(v):
                bars_with_seeds(ax, gi + (k - 1) * 0.26, v, 0.24, colour(cond))
    ax.set_xticks(range(4), [f"{'APTOS' if c == 'aptos361' else 'Messidor-2'}\nCLAHE {cl}" for c, cl, _ in groups6])
    ax.set_title(name)
    ax.set_ylim(0, 100 if m == "accuracy" else 1)
legend_below(fig, [Patch(color="#7F7F7F", label="No adapter (zero-shot, same 4-bit model)"),
                   Patch(color="#CC79A7", label="LoRA adapter, lr 5e-4"),
                   Patch(color="#0072B2", label="LoRA adapter, lr 2e-4 (lighter shade: CLAHE off)")], ncol=3)
save(fig, "F06", "adapter_vs_zero_shot", "With and without the LoRA adapter",
     "per_run_metrics.csv (training_aggregate.csv and messidor2_aggregate.csv for the CIs)",
     "The same 4-bit Qwen3-VL-8B base with no adapter (zero-shot) and with the LoRA adapter at lr 5e-4 and 2e-4, scored "
     "with the same five-answer likelihood scorer, on APTOS-361 and verified Messidor-2, CLAHE on and off. Bars: mean; "
     "dots: seeds.", f06)

## F07. CLAHE on vs off

In [ ]:
pairs = [("Qwen LoRA\nlr 2e-4", Q_ON_2E4, Q_OFF_2E4), ("Qwen LoRA\nlr 5e-4", Q_ON_5E4, Q_OFF_5E4),
         ("Qwen\nzero-shot", Z_ON, Z_OFF), ("RetiZero", R_ON, R_OFF)]
rows = []
for name, on, off in pairs:
    for cohort in ("aptos361", "messidor2"):
        for clahe, cond in (("on", on), ("off", off)):
            for v_acc, v_qwk in zip(metric(cond, cohort, "accuracy"), metric(cond, cohort, "qwk")):
                rows.append({"model": name.replace("\n", " "), "cohort": cohort, "clahe": clahe, "accuracy": v_acc, "qwk": v_qwk})
f07 = pd.DataFrame(rows)
fig, axes = plt.subplots(2, 2, figsize=(FULL_WIDTH, 4.8), sharex=True)
for i, (m, scale, name) in enumerate([("accuracy", 100, "Accuracy (%)"), ("qwk", 1, "QWK")]):
    for j, cohort in enumerate(("aptos361", "messidor2")):
        ax = axes[i, j]
        for k, (mname, on, off) in enumerate(pairs):
            d = f07[(f07.model == mname.replace("\n", " ")) & (f07.cohort == cohort)]
            means = [d[d.clahe == c][m].mean() * scale for c in ("off", "on")]
            ax.plot([k - 0.15, k + 0.15], means, color=colour(on), linewidth=1.3)
            for c, dx, face in (("off", -0.15, "white"), ("on", 0.15, colour(on))):
                v = d[d.clahe == c][m] * scale
                ax.scatter(np.full(len(v), k + dx), v, s=12, facecolor=face, edgecolor=colour(on), zorder=3)
        ax.set_xticks(range(len(pairs)), [p[0] for p in pairs])
        ax.set_ylabel(name)
        ax.set_title(f"{'Accuracy' if m == 'accuracy' else 'QWK'}, {COHORT[cohort]}", loc="left")
legend_below(fig, [Line2D([], [], marker="o", ls="", mfc="white", mec="black", label="CLAHE off (seed)"),
                   Line2D([], [], marker="o", ls="", mfc="black", mec="black", label="CLAHE on (seed)"),
                   Line2D([], [], color="black", label="line joins the two means")], ncol=3)
save(fig, "F07", "clahe_on_off", "CLAHE on vs off",
     "per_run_metrics.csv (clahe_aggregate.csv, rank_lr2e-4_aggregate.csv, messidor2_aggregate.csv for the CIs)",
     "Effect of CLAHE on each model, per seed (open: off; filled: on; lines join the means). No fine-tuned model shows a "
     "consistent CLAHE benefit; CLAHE lowers zero-shot Qwen accuracy.", f07)

## F08. Confusion matrices

In [ ]:
cells = [(Q_ON_2E4, "aptos361"), (R_OFF, "aptos361"), (Q_ON_2E4, "messidor2"), (R_OFF, "messidor2")]
fig, axes = plt.subplots(2, 2, figsize=(6.6, 6.8))
rows = []
for ax, (cond, cohort) in zip(axes.flat, cells):
    f = pooled(cond, cohort)
    cm = pd.crosstab(f.y_true, f.y_pred).reindex(index=range(5), columns=range(5), fill_value=0).to_numpy()
    norm = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{norm[i, j]:.0%}\n({cm[i, j]})", ha="center", va="center", fontsize=5.5,
                    color="white" if norm[i, j] > 0.55 else "black")
            rows.append({"condition": cond, "cohort": cohort, "true": i, "pred": j, "count": int(cm[i, j]), "row_share": norm[i, j]})
    ax.set_xticks(range(5), range(5)); ax.set_yticks(range(5), range(5)); ax.grid(False)
    ax.set_xlabel("Predicted grade"); ax.set_ylabel("True grade")
    model = "Qwen LoRA lr 2e-4" if cond.startswith("Qwen") else "RetiZero, CLAHE off"
    ax.set_title(f"{model}\n{COHORT[cohort]}, 3 seeds pooled", fontsize=7.5)
save(fig, "F08", "confusion_matrices", "Confusion matrices", "per-image predictions (bra.load_all)",
     "Row-normalised confusion matrices for the headline Qwen model (LoRA r16, lr 2e-4, CLAHE on) and RetiZero (CLAHE "
     "off), on APTOS-361 and verified Messidor-2; three seeds pooled; counts in brackets.", pd.DataFrame(rows))

## F09. How far off are the errors?

In [ ]:
es = pd.read_csv(TABLES / "error_structure.csv")
conds9 = [Q_ON_2E4, Q_ON_5E4, R_OFF, R_ON]
sel = [(c, k) for k in ("aptos361", "messidor", "messidor2") for c in conds9]
f09 = pd.concat([es[(es.condition == c) & (es.cohort == k)] for c, k in sel], ignore_index=True)
parts = [("exact", "Exact", "#4D9221"), ("off_by_1", "Off by 1 grade", "#B8E186"), ("off_by_2", "Off by 2 grades", "#F1B6DA"),
         ("off_by_3_or_4", "Off by 3 or 4 grades", "#C51B7D")]
tot = f09[[p for p, _, _ in parts]].sum(axis=1).to_numpy()
fig, ax = plt.subplots(figsize=(FULL_WIDTH, 0.24 * len(f09) + 1.2))
left = np.zeros(len(f09))
for p, name, col in parts:
    share = f09[p].to_numpy() / tot
    ax.barh(range(len(f09)), share, left=left, color=col, edgecolor="white", linewidth=0.4)
    for i, (s, l) in enumerate(zip(share, left)):
        if s >= 0.07:
            ax.text(l + s / 2, i, f"{s:.0%}", ha="center", va="center", fontsize=6)
    left += share
short_name = {Q_ON_2E4: "Qwen lr 2e-4", Q_ON_5E4: "Qwen lr 5e-4", R_OFF: "RetiZero, CLAHE off", R_ON: "RetiZero, CLAHE on"}
ax.set_yticks(range(len(f09)), [f"{short_name[c]} | {COHORT[k]}" for c, k in zip(f09.condition, f09.cohort)])
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.grid(False); ax.set_xlabel("Share of images (three seeds pooled)")
legend_below(fig, [Patch(color=c, label=n) for _, n, c in parts], ncol=4)
save(fig, "F09", "error_distance", "Error distance", "error_structure.csv",
     "Share of predictions that are exact, or off by one, two, or three or more grades (shares below 7% not labelled). "
     "Most errors are between adjacent grades, which is why QWK stays high while accuracy is moderate.", f09)

## F10. Class-wise sensitivity and specificity

In [ ]:
cw = pd.read_csv(TABLES / "classwise_condition_mean.csv")
f10 = cw[cw.condition.isin([Q_ON_2E4, R_OFF]) & cw.cohort.isin(["aptos361", "messidor2"])]
fig, axes = plt.subplots(2, 2, figsize=(FULL_WIDTH, 4.6), sharey=True, sharex=True)
for i, cohort in enumerate(("aptos361", "messidor2")):
    for j, m in enumerate(("sensitivity", "specificity")):
        ax = axes[i, j]
        for k, cond in enumerate((Q_ON_2E4, R_OFF)):
            d = f10[(f10.condition == cond) & (f10.cohort == cohort)].sort_values("grade")
            ax.bar(d.grade + (k - 0.5) * 0.38, d[f"{m}_mean"], 0.36, yerr=d[f"{m}_std"], capsize=1.5, color=colour(cond))
        ax.set_xticks(range(5), [g.replace(" ", "\n", 1) for g in GRADE]); ax.set_ylim(0, 1.05)
        ax.set_title(f"{m.capitalize()}, {COHORT[cohort]}", loc="left")
legend_below(fig, [Patch(color=colour(c), label=label(c)) for c in (Q_ON_2E4, R_OFF)], ncol=2)
save(fig, "F10", "classwise_sensitivity_specificity", "Class-wise sensitivity and specificity",
     "classwise_condition_mean.csv",
     "One-vs-rest sensitivity and specificity per grade (mean ± SD over three seeds) for Qwen (lr 2e-4, CLAHE on) and "
     "RetiZero (CLAHE off).", f10)

## F11. One-vs-rest ROC curves per grade (ROC-AUC comments)

Each seed is one thin line. The number in each panel is the mean per-seed AUC, the same value as in the tables.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(FULL_WIDTH, 3.6), sharex=True, sharey=True)
rows = []
for i, cohort in enumerate(("aptos361", "messidor2")):
    for g in range(5):
        ax = axes[i, g]
        ax.plot([0, 1], [0, 1], color="#CCCCCC", linewidth=0.6)
        text = []
        for cond, short_label in ((Q_ON_2E4, "Qwen"), (R_OFF, "RetiZero")):
            aucs = []
            for r in runs_of(cond, cohort):
                f = r["frame"]
                fpr, tpr = roc((f.y_true.to_numpy() == g).astype(int), f[f"p{g}"].to_numpy(float))
                ax.plot(fpr, tpr, color=colour(cond), linewidth=0.7, alpha=0.85)
                aucs.append(r["metrics"][f"auc_g{g}"])
            rows.append({"cohort": cohort, "grade": g, "condition": cond, "mean_auc": np.mean(aucs), "seed_aucs": aucs})
            text.append(f"{short_label} {np.mean(aucs):.2f}")
        ax.text(0.97, 0.04, "AUC\n" + "\n".join(text), transform=ax.transAxes, ha="right", va="bottom", fontsize=5.5)
        ax.set_aspect("equal")
        ax.set_xticks([0, 0.5, 1], ["0", "0.5", "1"]); ax.set_yticks([0, 0.5, 1], ["0", "0.5", "1"])
        if i == 0:
            ax.set_title(GRADE[g], fontsize=7.5)
    axes[i, 0].set_ylabel(f"{'APTOS-361' if cohort == 'aptos361' else 'Messidor-2'}\nsensitivity")
for ax in axes[1]:
    ax.set_xlabel("1 − specificity", fontsize=7)
legend_below(fig, [Line2D([], [], color=colour(Q_ON_2E4), label="Qwen LoRA, lr 2e-4, CLAHE on"),
                   Line2D([], [], color=colour(R_OFF), label="RetiZero, CLAHE off")], ncol=2)
save(fig, "F11", "roc_per_grade", "One-vs-rest ROC curves per grade",
     "per-image predictions (bra.load_all); auc_g* in per_run_metrics.csv",
     "One-vs-rest ROC curve for each DR grade, computed from each grade's own probability column, for Qwen (lr 2e-4, "
     "CLAHE on) and RetiZero (CLAHE off); one line per seed; numbers are the mean per-seed AUC.", pd.DataFrame(rows))

## F12. Referable DR: the operating point the models choose (limitation)

Referable DR is grade 2 or above. Its score is p2 + p3 + p4. Each dot marks the model's own operating point: predicted grade ≥ 2.

In [ ]:
conds12 = [Q_ON_2E4, R_OFF, R_APP, Z_ON]
fig, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 3.6))
rows = []
for ax, cohort in zip(axes, ("aptos361", "messidor2")):
    ax.plot([0, 1], [0, 1], color="#CCCCCC", linewidth=0.6)
    for cond in conds12:
        for r in runs_of(cond, cohort):
            f = r["frame"]
            y = (f.y_true.to_numpy() >= 2).astype(int)
            fpr, tpr = roc(y, f[["p2", "p3", "p4"]].sum(axis=1).to_numpy(float))
            pred = f.y_pred.to_numpy() >= 2
            sens, spec = pred[y == 1].mean(), (~pred[y == 0]).mean()
            ax.plot(fpr, tpr, color=colour(cond), linewidth=0.8, alpha=0.85)
            ax.scatter(1 - spec, sens, s=24, color=colour(cond), edgecolor="black", linewidth=0.5, zorder=4)
            rows.append({"cohort": cohort, "condition": cond, "seed": r["seed"], "referable_auc": r["metrics"]["referable_auc"],
                         "sensitivity_at_model_threshold": sens, "specificity_at_model_threshold": spec})
    ax.set_title(COHORT[cohort], loc="left"); ax.set_xlabel("1 − specificity"); ax.set_ylabel("Sensitivity")
    ax.set_aspect("equal")
legend_below(fig, [Line2D([], [], color=colour(c), marker="o", mec="black", mew=0.5, label=label(c)) for c in conds12], ncol=2)
save(fig, "F12", "referable_dr_operating_points", "Referable DR: ROC and operating points",
     "per-image predictions (bra.load_all); referable_* in per_run_metrics.csv",
     "ROC curves for referable DR (grade ≥ 2; score p2 + p3 + p4), one line per seed; dots mark each model's own "
     "operating point (predicted grade ≥ 2). On Messidor-2 the fine-tuned models operate at high specificity (about "
     "0.97) and low sensitivity (about 0.3), well below what their ROC curves allow; a recalibrated threshold would be "
     "needed for screening.", pd.DataFrame(rows))

## F13. Mild vs Normal, the main failure on Messidor

In [ ]:
mv = pd.read_csv(TABLES / "mild_vs_normal.csv")
conds13 = [Q_ON_2E4, Q_ON_5E4, R_OFF, R_ON]
coh13 = ["aptos361", "messidor", "messidor2"]
f13 = mv[mv.condition.isin(conds13) & mv.cohort.isin(coh13)]
panels = [("mild_predicted_normal_rate", "Mild eyes called Normal"), ("normal_predicted_mild_rate", "Normal eyes called Mild"),
          ("auc_mild_vs_normal", "Mild-vs-Normal AUC")]
fig, axes = plt.subplots(1, 3, figsize=(FULL_WIDTH, 3.0))
for ax, (m, name) in zip(axes, panels):
    for ci, cohort in enumerate(coh13):
        for k, cond in enumerate(conds13):
            v = f13[(f13.condition == cond) & (f13.cohort == cohort)][m]
            if len(v):
                bars_with_seeds(ax, ci + (k - 1.5) * 0.2, v, 0.18, colour(cond))
    ax.set_xticks(range(3), [SHORT_COHORT[c] for c in coh13]); ax.set_ylim(0, 1); ax.set_title(name)
    if m == "auc_mild_vs_normal":
        ax.axhline(0.5, color="black", linewidth=0.6, linestyle="--")
legend_below(fig, [Patch(color=colour(c), label=label(c)) for c in conds13]
             + [Line2D([], [], color="black", linestyle="--", linewidth=0.6, label="chance (AUC 0.5)")], ncol=3)
save(fig, "F13", "mild_vs_normal", "Mild vs Normal across cohorts", "mild_vs_normal.csv",
     "Mild (grade 1) eyes predicted Normal, Normal eyes predicted Mild, and the AUC separating Mild from Normal, on "
     "APTOS and both Messidor cohorts (bars: mean; dots: seeds; RetiZero with CLAHE was not scored on archived "
     "Messidor). Mild-vs-Normal discrimination falls to chance on Messidor; the cause is suspected domain shift, not "
     "confirmed by a grader.", f13)

## F14. Predicted vs true grade distribution

In [ ]:
conds14 = [Q_ON_2E4, R_OFF, R_ON, Z_ON]
rows = []
for cohort in ("aptos361", "messidor2"):
    true_share = runs_of(Q_ON_2E4, cohort)[0]["frame"]["y_true"].value_counts(normalize=True).reindex(range(5), fill_value=0)
    rows += [{"cohort": cohort, "series": "True grades", "grade": g, "share": true_share[g]} for g in range(5)]
    for cond in conds14:
        d = per_run[(per_run.condition == cond) & (per_run.cohort == cohort)]
        share = d[[f"n_pred_g{g}" for g in range(5)]].div(d["n"], axis=0).mean()
        rows += [{"cohort": cohort, "series": cond, "grade": g, "share": share[f"n_pred_g{g}"]} for g in range(5)]
f14 = pd.DataFrame(rows)
series = ["True grades"] + conds14
fig, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 3.0), sharey=True)
for ax, cohort in zip(axes, ("aptos361", "messidor2")):
    for k, s in enumerate(series):
        d = f14[(f14.cohort == cohort) & (f14.series == s)]
        ax.bar(d.grade + (k - 2) * 0.16, d.share, 0.15, color="black" if s == "True grades" else colour(s))
    ax.set_xticks(range(5), [g.replace(" ", "\n", 1) for g in GRADE]); ax.set_title(COHORT[cohort], loc="left")
axes[0].set_ylabel("Share of images")
legend_below(fig, [Patch(color="black", label="True grades")] + [Patch(color=colour(c), label=label(c)) for c in conds14], ncol=3)
save(fig, "F14", "predicted_vs_true_distribution", "Predicted vs true grade distribution",
     "per_run_metrics.csv (n_pred_g*); per-image labels",
     "Share of images per grade: true labels (black) against each model's predictions (mean over seeds). On Messidor-2 "
     "the fine-tuned models predict too few moderate and severe cases, i.e. they under-grade.", f14)

## F15. Calibration

In [ ]:
def reliability(f, bins=bra.ECE_BINS):  # same bins as the ECE in the tables
    p = f[[f"p{g}" for g in range(5)]].to_numpy(float)
    y = f.y_true.to_numpy(int)
    conf, correct = p.max(1), (p.argmax(1) == y)
    idx = np.clip(np.digitize(conf, np.linspace(0, 1, bins + 1)[1:-1], right=True), 0, bins - 1)
    return pd.DataFrame({"bin": range(bins), "n": [int((idx == b).sum()) for b in range(bins)],
                         "confidence": [conf[idx == b].mean() if (idx == b).any() else np.nan for b in range(bins)],
                         "accuracy": [correct[idx == b].mean() if (idx == b).any() else np.nan for b in range(bins)]})


conds15 = [Q_ON_2E4, Q_ON_5E4, R_OFF, R_ON]
fig, axes = plt.subplots(1, 3, figsize=(FULL_WIDTH, 3.0))
rows = []
for ax, cohort in zip(axes[:2], ("aptos361", "messidor2")):
    ax.plot([0, 1], [0, 1], color="#CCCCCC", linewidth=0.6)
    for cond in conds15:
        rel = reliability(pooled(cond, cohort)).assign(condition=cond, cohort=cohort)
        rows.append(rel)
        ok = rel[rel.n >= 10]
        ax.plot(ok.confidence, ok.accuracy, marker="o", markersize=2.5, linewidth=0.9, color=colour(cond))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("Confidence"); ax.set_title(f"Reliability, {SHORT_COHORT[cohort].replace(chr(10), ' ')}", loc="left")
axes[0].set_ylabel("Accuracy")
for ci, cohort in enumerate(("aptos361", "messidor2")):
    for k, cond in enumerate(conds15):
        bars_with_seeds(axes[2], ci + (k - 1.5) * 0.2, metric(cond, cohort, "ece"), 0.18, colour(cond))
axes[2].set_xticks([0, 1], ["APTOS-361", "Messidor-2"]); axes[2].set_title("Expected calibration error", loc="left")
legend_below(fig, [Patch(color=colour(c), label=label(c)) for c in conds15], ncol=2)
save(fig, "F15", "calibration", "Calibration",
     "per-image predictions (bra.load_all); ece in per_run_metrics.csv",
     "Reliability diagrams (top-probability confidence in 15 equal-width bins, the same bins as the ECE, seeds pooled, bins with at least 10 images) and "
     "expected calibration error per seed. Both models are over-confident on Messidor-2.", pd.concat(rows, ignore_index=True))

## F16. Computational efficiency

Every number was measured on the same PC: one RTX 5070 Ti 16 GB. Log scales are used where the models differ by more than 20 times.

In [ ]:
eff = pd.read_csv(TABLES / "efficiency_final.csv")


def pick(quantity, model_start, model_contains=""):
    d = eff[(eff.quantity == quantity) & eff.model.str.startswith(model_start) & eff.model.str.contains(model_contains, regex=False)]
    if d.empty:
        raise KeyError(f"{quantity} / {model_start} / {model_contains}")
    sd = d.iloc[0]["sd"]
    return float(d.iloc[0]["mean"]), float(sd) if sd == sd else 0.0


QWEN, RETI = "Qwen3-VL-8B + LoRA r16, 4-bit", "RetiZero ViT-L/16"
items = [
    ("Trainable parameters\n(millions)", pick("Trainable parameters", QWEN), pick("Trainable parameters", RETI), 1e-6),
    ("Total parameters\n(millions)", pick("Total parameters", QWEN), pick("Total parameters", RETI), 1e-6),
    ("Training time,\none full run (min)", pick("Training wall time, final attempt (from 5 s utilisation samples)", QWEN, "lr2e-4"),
     pick("Training wall time, all attempts", RETI, "all weights trained"), 1 / 60),
    ("Inference per image\n(ms, batch 1)", pick("Scoring warm median time per image (batch 1)", QWEN),
     pick("Scoring warm median time per image (batch 1)", RETI), 1000),
    ("Peak training GPU\nmemory (GiB)", pick("Training peak GPU memory, PyTorch allocated", QWEN),
     pick("Training peak GPU memory, PyTorch allocated", RETI, "all weights trained"), 1),
    ("Peak inference GPU\nmemory (GiB)", pick("Scoring peak GPU memory, PyTorch allocated (batch 1)", QWEN),
     pick("Scoring peak GPU memory, PyTorch allocated (batch 1)", RETI), 1),
]
f16 = pd.DataFrame([{"quantity": q.replace("\n", " "), "qwen_mean": qm * s, "qwen_sd": qs * s, "retizero_mean": rm * s,
                     "retizero_sd": rs * s} for q, (qm, qs), (rm, rs), s in items])
tim = pd.read_csv(TABLES / "timing_and_energy_final_02.csv")
tim = tim[tim.job.str.startswith("timing_")].copy()


def config_name(job):
    c = job.replace("timing_", "")
    m = re.match(r"r(\d+)_(on|off)", c)
    lr = re.search(r"lr0p000(\d)", c)
    alpha = re.search(r"alpha(\d+)", c)
    name = f"r{m.group(1)}" + (f" α{alpha.group(1)}" if alpha else "") + f", CLAHE {m.group(2)}, lr {lr.group(1) + 'e-4' if lr else '5e-4'}"
    return name + (", previous split" if "previous_split" in c else "")


tim["config"] = tim.job.map(config_name)
tim = tim.sort_values("config")
fig = plt.figure(figsize=(FULL_WIDTH, 7.4))
top, bottom = fig.subfigures(2, 1, height_ratios=[1.35, 1])
axes = top.subplots(2, 3)
for ax, (q, (qm, qs), (rm, rs), s) in zip(axes.flat, items):
    vals, errs = [qm * s, rm * s], [qs * s, rs * s]
    ax.bar([0, 1], vals, yerr=errs, color=["#0072B2", "#E69F00"], capsize=2, width=0.6)
    log = max(vals) / max(min(vals), 1e-9) > 20
    if log:
        ax.set_yscale("log")
        ax.set_ylim(min(vals) / 2, max(vals) * 6)
    else:
        ax.set_ylim(0, max(v + e for v, e in zip(vals, errs)) * 1.3)
    ax.set_xticks([0, 1], ["Qwen", "RetiZero"])
    ax.set_title(q, fontsize=7.5)
    for x0, v, e in zip((0, 1), vals, errs):
        ax.text(x0, (v + e) * (1.25 if log else 1) + (0 if log else max(vals) * 0.03),
                f"{v:,.0f}" if v >= 100 else f"{v:.3g}", ha="center", va="bottom", fontsize=6)
a1, a2 = bottom.subplots(1, 2)
yy = np.arange(len(tim))
a1.barh(yy, tim.median_s_per_image * 1000, color="#0072B2")
a1.set_yticks(yy, tim.config); a1.invert_yaxis(); a1.set_xlabel("Inference, ms per image")
a1.set_title("Qwen inference time across configurations", loc="left")
a2.barh(yy, tim.energy_wh_per_image.astype(float), color="#56B4E9")
a2.set_yticks(yy, [""] * len(yy)); a2.invert_yaxis(); a2.set_xlabel("Energy, Wh per image (whole card)")
a2.set_title("Energy per image", loc="left")
save(fig, "F16", "efficiency", "Computational efficiency", "efficiency_final.csv, timing_and_energy_final_02.csv",
     "Top: model size, training time for one full run, inference time per image (batch 1, warm median) and peak GPU "
     "memory for Qwen3-VL-8B + LoRA r16 (4-bit) and RetiZero ViT-L/16 (all weights trained), measured on the same "
     "RTX 5070 Ti 16 GB. Bottom: Qwen inference time and whole-card energy per image for seven configurations, which do "
     "not differ.", pd.concat([f16.assign(panel="top"), tim.assign(panel="bottom")], ignore_index=True))

## F17. Seed-to-seed variability of every APTOS condition

In [ ]:
d17 = per_run[(per_run.cohort == "aptos361") & per_run.seed.notna()].copy()
order17 = d17.groupby("condition").accuracy.mean().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(FULL_WIDTH, 0.2 * len(order17) + 1.3))
for i, cond in enumerate(order17):
    v = d17[d17.condition == cond]
    lr = re.search(r"lr(\de-\d)", cond)
    col = "#E69F00" if cond.startswith("RetiZero") else LR_COLOUR.get(lr.group(1), "#444444")
    ok, bad = v[~failed(v.n_grades_predicted)], v[failed(v.n_grades_predicted)]
    ax.plot([v.accuracy.min() * 100, v.accuracy.max() * 100], [i, i], color=col, linewidth=0.8)
    ax.scatter(ok.accuracy * 100, np.full(len(ok), i), s=12, color=col, zorder=3)
    ax.scatter(bad.accuracy * 100, np.full(len(bad), i), s=24, marker="x", color="#D55E00", zorder=3)
ax.set_yticks(range(len(order17)), order17, fontsize=6.5)
ax.set_xlabel("APTOS-361 accuracy (%)"); ax.set_xlim(0, 100)
legend_below(fig, [Line2D([], [], marker="o", ls="", color=c, label=f"Qwen lr {k}") for k, c in LR_COLOUR.items()]
             + [Line2D([], [], marker="o", ls="", color="#E69F00", label="RetiZero"),
                Line2D([], [], marker="x", ls="", color="#D55E00", label="failed run (≤ 3 grades)")], ncol=5)
save(fig, "F17", "seed_variability", "Seed-to-seed variability", "per_run_metrics.csv",
     "Accuracy of every seed of every fine-tuned condition on APTOS-361, sorted by mean; lines span the seeds. Runs at "
     "lr 5e-4 vary widely and include failed runs; at lr 2e-4 and for RetiZero the spread is a few points.",
     d17[["condition", "seed", "accuracy", "qwk", "n_grades_predicted"]])

## F18. Near-duplicate sensitivity

In [ ]:
nd = pd.read_csv(TABLES / "near_duplicate_sensitivity.csv")
f18 = nd.groupby(["condition", "threshold", "excluded_test_images"], as_index=False)[["accuracy_all", "accuracy_kept", "qwk_all", "qwk_kept"]].mean()
steps = sorted(f18.threshold.unique())
xt = ["All 361"] + [f"pHash ≤ {int(t)}\n({int(f18[f18.threshold == t].excluded_test_images.iloc[0])} removed)" for t in steps]
fig, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 2.9))
for ax, (m, scale, name) in zip(axes, [("accuracy", 100, "Accuracy (%)"), ("qwk", 1, "QWK")]):
    for cond, g in f18.groupby("condition"):
        g = g.sort_values("threshold")
        ys = [g[f"{m}_all"].iloc[0] * scale] + list(g[f"{m}_kept"] * scale)
        ax.plot(range(len(ys)), ys, marker="o", markersize=3, color=colour(cond))
    ax.set_xticks(range(len(xt)), xt); ax.set_title(name, loc="left")
legend_below(fig, [Line2D([], [], color=colour(c), marker="o", markersize=3, label=label(c)) for c in f18.condition.unique()], ncol=3)
save(fig, "F18", "near_duplicate_sensitivity", "Near-duplicate sensitivity", "near_duplicate_sensitivity.csv",
     "Mean accuracy and QWK on all 361 test images and after removing test images with a perceptual-hash near-duplicate "
     "in training at distance 0, ≤ 2 and ≤ 4.", f18)

## F19. Archived vs verified Messidor

In [ ]:
av = pd.read_csv(TABLES / "messidor_archived_vs_verified.csv")


def span(s):
    return f"{s.min():,}" if s.min() == s.max() else f"{s.min():,}–{s.max():,}"


fig, (a, b) = plt.subplots(1, 2, figsize=(FULL_WIDTH, 3.4))
for model, col in (("Qwen", "#0072B2"), ("RetiZero", "#E69F00")):
    d = av[av.model == model]
    a.scatter(d.accuracy_archived * 100, d.accuracy_verified_same_images * 100, s=16, color=col)
    b.scatter(d.accuracy_archived * 100, d.accuracy_verified_all_1744 * 100, s=16, color=col)
for ax, t, yl in ((a, "A  Same 1,057 images, rebuilt inputs", "Accuracy, rebuilt inputs (%)"),
                  (b, "B  All 1,744 verified images", "Accuracy, verified Messidor-2 (%)")):
    ax.plot([20, 65], [20, 65], color="#CCCCCC", linewidth=0.7)
    ax.set_xlim(20, 65); ax.set_ylim(20, 65); ax.set_aspect("equal")
    ax.set_xlabel("Accuracy, archived inputs (%)"); ax.set_ylabel(yl); ax.set_title(t, loc="left")
legend_below(fig, [Line2D([], [], marker="o", ls="", color="#0072B2",
                          label=f"Qwen runs (same grade on {span(av[av.model == 'Qwen'].same_predicted_grade)} of 1,057 images)"),
                   Line2D([], [], marker="o", ls="", color="#E69F00",
                          label=f"RetiZero runs (same grade on {span(av[av.model == 'RetiZero'].same_predicted_grade)} of 1,057)")], ncol=1)
save(fig, "F19", "archived_vs_verified_messidor", "Archived vs verified Messidor", "messidor_archived_vs_verified.csv",
     "(A) Accuracy of each run on the archived Messidor inputs against the inputs rebuilt from the raw photos, same "
     "1,057 images: Qwen is identical; RetiZero differs slightly because its archived route went raw → 512 → 224. "
     "(B) Archived accuracy against accuracy on the full verified Messidor-2 cohort.", av)

## Training curves from the run logs (F20–F22)

Each training run saved its full history in `train_result.json`:

- **Qwen:** the Hugging Face trainer log, with a training-loss point every 5 optimiser steps and a validation loss after each evaluation. The checkpoint with the lowest validation loss was kept.
- **RetiZero:** one row per epoch with training loss, validation loss and validation accuracy. The epoch with the highest validation accuracy was kept.

The runs are the same ones as in the tables: each run's training folder is found from the predictions file it was scored from.

In [ ]:
def train_folder(r):
    """The training folder of a run: next to its APTOS predictions (Qwen), one level above evaluation/ (RetiZero),
    or, for a run re-scored in scores/<job>__<cohort>/ (the FP32 fallback), train/<job>/ of the same campaign."""
    p = r["path"]
    for d in (p.parent, p.parent.parent, p.parent.parent.parent / "train" / r["name"].split("__")[0]):
        if (d / "train_result.json").is_file():
            return d
    return None


def training_history(r):
    """Long-format history of one training run: kind (train / validation / validation accuracy), epoch, value."""
    d = train_folder(r)
    rec = json.loads((d / "train_result.json").read_text(encoding="utf-8"))
    rows = []
    if r["model"] == "Qwen":
        train, val = {}, {}
        for e in rec["history"]:             # keep the last entry per step (a resumed attempt repeats steps)
            if "loss" in e:
                train[e["step"]] = (e["epoch"], e["loss"])
            if "eval_loss" in e:
                val[e["step"]] = (e["epoch"], e["eval_loss"])
        rows += [{"kind": "train_loss", "step": s, "epoch": ep, "value": v} for s, (ep, v) in sorted(train.items())]
        rows += [{"kind": "val_loss", "step": s, "epoch": ep, "value": v} for s, (ep, v) in sorted(val.items())]
    else:
        for e in rec["history"]:
            rows += [{"kind": "train_loss", "step": e["epoch"], "epoch": e["epoch"], "value": e["train_loss"]},
                     {"kind": "val_loss", "step": e["epoch"], "epoch": e["epoch"], "value": e["validation_loss"]},
                     {"kind": "val_accuracy", "step": e["epoch"], "epoch": e["epoch"], "value": e["validation_accuracy"]}]
    best = rec.get("best_epoch")
    return pd.DataFrame(rows).assign(condition=r["condition"], seed=r["seed"], job=r["name"], folder=d.name,
                                     best_epoch=best)


trained = {}
for r in sorted(runs, key=lambda r: r["cohort"] != "aptos361"):   # name each training run after its APTOS-361 scoring
    d = train_folder(r)
    if d is not None and d not in trained:
        trained[d] = r
histories = pd.concat([training_history(r) for r in trained.values()], ignore_index=True)
failed_jobs = set(cv.loc[cv.failed, "job"])
histories["failed_run"] = histories.job.isin(failed_jobs)
histories.to_csv(OUT / "training_histories_all_runs.csv", index=False)
n_qwen = histories[histories.condition.str.startswith("Qwen")].folder.nunique()
n_rz = histories[histories.condition.str.startswith("RetiZero")].folder.nunique()
n_failed = histories.job[histories.failed_run].nunique()
print(f"Training histories: {n_qwen} Qwen runs, {n_rz} RetiZero runs; {n_failed} failed runs "
      f"(collapse_vs_validation.csv lists {cv.failed.sum()} of {len(cv)} fine-tuned Qwen runs as failed)")
assert n_qwen == len(cv) and n_failed == cv.failed.sum(), "a fine-tuned run has no training history"
REPORT.append(f"- Training histories read for {n_qwen} Qwen and {n_rz} RetiZero runs")


def smooth(values, window=9):
    """Centered running median, so single noisy batches do not hide the trend. Raw values are in the data CSV."""
    return pd.Series(values).rolling(window, center=True, min_periods=1).median().to_numpy()


def log_y(ax, values):
    """Log y axis with plain-number ticks, so one spike does not flatten every other curve."""
    lo, hi = np.nanmin(values), np.nanmax(values)
    ax.set_yscale("log")
    ax.set_ylim(lo / 1.15, hi * 1.15)
    ticks = [t for t in (0.005, 0.0075, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.5, 1, 2) if lo / 1.15 <= t <= hi * 1.15]
    ax.yaxis.set_major_locator(matplotlib.ticker.FixedLocator(ticks))
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())


def integer_x(ax):
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))

## F20. Qwen training and validation loss by learning rate

In [ ]:
F20_CONDS = [("1e-4", "Qwen r16 CLAHE-on lr1e-4"), ("2e-4", Q_ON_2E4), ("5e-4", Q_ON_5E4)]
f20 = histories[histories.condition.isin([c for _, c in F20_CONDS])].copy()
f20["lr"] = f20.condition.map({c: lr for lr, c in F20_CONDS})
fig, (a, b) = plt.subplots(1, 2, figsize=(FULL_WIDTH, 2.9))
for lr, cond in F20_CONDS:
    for (job, seed), d in f20[f20.condition == cond].groupby(["job", "seed"]):
        bad = d.failed_run.iloc[0]
        tr, va = d[d.kind == "train_loss"], d[d.kind == "val_loss"]
        style = dict(color=LR_COLOUR[lr], lw=0.9, ls="--" if bad else "-", alpha=0.9)
        a.plot(tr.epoch, smooth(tr.value), **style)
        b.plot(va.epoch, va.value, marker="x" if bad else "o", ms=3, **style)
        kept = va.loc[va.value.idxmin()]
        b.scatter(kept.epoch, kept.value, s=26, color=LR_COLOUR[lr], edgecolor="black", linewidth=0.7, zorder=4)
a.set_yscale("log"); a.set_xlabel("Epoch"); a.set_ylabel("Training loss (log scale)")
a.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
a.set_title("A  Training loss (running median of 9 log points)", loc="left")
log_y(b, f20[f20.kind == "val_loss"].value)
b.set_xlabel("Epoch"); b.set_ylabel("Validation loss (log scale)")
b.set_title("B  Validation loss at each evaluation", loc="left")
integer_x(b)
legend_below(fig, [Line2D([], [], color=c, label=f"lr {k}") for k, c in LR_COLOUR.items()]
             + [Line2D([], [], color="black", ls="--", marker="x", ms=3.5, label="failed run (≤ 3 grades)"),
                Line2D([], [], marker="o", ls="", color="white", markeredgecolor="black", label="kept checkpoint")], ncol=5)
save(fig, "F20", "qwen_training_curves_by_lr", "Qwen training and validation loss by learning rate",
     "train_result.json of each run (Hugging Face trainer history); failed runs from collapse_vs_validation.csv",
     "Training loss (A, running median of the trainer log) and validation loss after each epoch (B, log scale) of Qwen "
     "rank 16, CLAHE on, at three learning rates, three seeds each. Training stops when the validation loss has not "
     "improved for two evaluations, so runs differ in length, and the checkpoint with the lowest validation loss is kept "
     "(outlined dot). The failed run (dashed, lr 5e-4) never reaches the validation loss of any working run, as in F05A. "
     "The two large spikes (one lr 2e-4 and one lr 5e-4 seed) come after those runs' kept checkpoints. Mean kept "
     "validation loss by learning rate is in F05B.",
     f20[["condition", "lr", "seed", "job", "kind", "step", "epoch", "value", "failed_run"]])

## F21. RetiZero training curves

In [ ]:
RZ_GROUPS = [("matched", "Clean split, CLAHE off (native, headline)", "#E69F00"),
             ("clahe", "Clean split, CLAHE on", "#D55E00"),
             ("historical", "Original split (replay of the paper's training)", "#8C8C8C")]
f21 = histories[histories.folder.str.match(r"^(matched|clahe|historical)_t\d+$")].copy()
f21["group"] = f21.folder.str.extract(r"^([a-z]+)_t")[0]
fig, axes = plt.subplots(1, 3, figsize=(FULL_WIDTH, 2.6))
for ax, (kind, name, scale) in zip(axes, [("train_loss", "Training loss", 1), ("val_loss", "Validation loss", 1),
                                         ("val_accuracy", "Validation accuracy (%)", 100)]):
    for key, _, col in RZ_GROUPS:
        for folder, d in f21[(f21.group == key) & (f21.kind == kind)].groupby("folder"):
            ax.plot(d.epoch, d.value * scale, color=col, lw=0.9, alpha=0.9)
            best = d[d.epoch == d.best_epoch]
            ax.scatter(best.epoch, best.value * scale, s=14, color=col, edgecolor="black", linewidth=0.5, zorder=3)
    ax.set_xlabel("Epoch"); ax.set_title(name)
    integer_x(ax)
legend_below(fig, [Line2D([], [], color=c, label=n) for _, n, c in RZ_GROUPS]
             + [Line2D([], [], marker="o", ls="", color="white", markeredgecolor="black", label="kept epoch (best validation accuracy)")], ncol=2)
save(fig, "F21", "retizero_training_curves", "RetiZero training curves",
     "train_result.json of each RetiZero run (retizero_01/train)",
     "RetiZero (all weights trained) per epoch: training loss, validation loss and validation accuracy, three seeds per "
     "setting. Training stops after four epochs without a higher validation accuracy (at most 15 epochs) and the epoch "
     "with the highest validation accuracy is kept (outlined dot). Validation loss reaches its minimum by epoch 2 to 7 "
     "and then rises while training loss keeps falling.",
     f21[["condition", "group", "seed", "folder", "kind", "epoch", "value", "best_epoch"]])

## F22. Validation loss across the LoRA rank and alpha grid at lr 2e-4

In [ ]:
RANK_COLOUR = {8: "#009E73", 16: "#0072B2", 32: "#D55E00"}
grid = []
for rank in (8, 16, 32):
    for clahe in ("on", "off"):
        grid.append((rank, clahe, "16", f"Qwen r{rank} CLAHE-{clahe} lr2e-4"))
        if rank != 16:
            grid.append((rank, clahe, "= rank", f"Qwen r{rank} CLAHE-{clahe} lr2e-4 alpha{rank}"))
f22 = []
for rank, clahe, alpha, cond in grid:
    d = histories[(histories.condition == cond) & (histories.kind == "val_loss")]
    f22.append(d.assign(rank=rank, clahe=clahe, alpha=alpha))
f22 = pd.concat(f22, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 2.8), sharey=True)
for ax, clahe in zip(axes, ("on", "off")):
    for (rank, alpha, job), d in f22[f22.clahe == clahe].groupby(["rank", "alpha", "job"]):
        ax.plot(d.epoch, d.value, color=RANK_COLOUR[rank], lw=0.9, ls="-" if alpha == "16" else ":", marker="o", ms=2.5)
    ax.set_xlabel("Epoch"); ax.set_title(f"CLAHE {clahe}")
    integer_x(ax)
log_y(axes[0], f22.value)
axes[0].set_ylabel("Validation loss (log scale)")
legend_below(fig, [Line2D([], [], color=c, label=f"rank {r}") for r, c in RANK_COLOUR.items()]
             + [Line2D([], [], color="black", ls="-", label="alpha 16"), Line2D([], [], color="black", ls=":", label="alpha = rank")], ncol=5)
save(fig, "F22", "lora_grid_validation_curves", "Validation loss across the LoRA grid at lr 2e-4",
     "train_result.json of each run (final_01 and final_02 train folders)",
     "Validation loss after each evaluation for every adapter in the lr 2e-4 grid (rank 8, 16, 32; alpha 16 or alpha = "
     "rank; CLAHE on and off; three seeds each; log scale). Every run reaches a validation loss in the same narrow range, "
     "consistent with the absence of any significant rank or alpha effect on the test set (F04). Later rises, the largest "
     "a rank-16 CLAHE-on seed at epoch 5, do not change the result: each run keeps its lowest-validation-loss checkpoint.",
     f22[["condition", "rank", "alpha", "clahe", "seed", "job", "epoch", "step", "value"]])

## F23–F25. Methodology and architecture diagrams

Schematics, not data plots. Every number in them comes from the code or the run records: the split counts from the
duplicate audit, the parameter counts from the safetensors headers (`parameter_counts.json`), the token count from the
scoring records (328 prompt tokens, 5 forward passes per image), and the training settings from each run's saved
configuration.

In [ ]:
from matplotlib.patches import FancyBboxPatch
import textwrap

QWEN_FILL, QWEN_EDGE = "#DCEBF7", "#0072B2"
RZ_FILL, RZ_EDGE = "#FBEBCB", "#B07800"
SHARED_FILL, SHARED_EDGE = "#EFEFEF", "#555555"
TRAIN_FILL, TRAIN_EDGE = "#DDF2E6", "#009E73"
CHAR_W = 0.056   # inches per character at 7 pt, used to wrap text inside boxes


def canvas(w, h):
    fig = plt.figure(figsize=(w, h), layout=None)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, w); ax.set_ylim(0, h); ax.axis("off")
    return fig, ax


def box(ax, x, y, w, h, text, fill=SHARED_FILL, edge=SHARED_EDGE, title=None, size=7, dashed=False):
    """A rounded box with wrapped, centred text; an optional bold first line."""
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.06", facecolor=fill,
                                edgecolor=edge, linewidth=0.9, linestyle="--" if dashed else "-"))
    width = max(int((w - 0.14) / (CHAR_W * size / 7)), 8)
    body = "\n".join(textwrap.fill(part, width) for part in text.split("\n"))
    line = size * 1.3 / 72                      # line height in inches
    n = body.count("\n") + 1 + (1 if title else 0)
    top = y + h / 2 + n * line / 2
    if title:
        ax.text(x + w / 2, top - line / 2, title, ha="center", va="center", fontsize=size, fontweight="bold")
        top -= line
    ax.text(x + w / 2, top - (n - (1 if title else 0)) * line / 2, body, ha="center", va="center", fontsize=size,
            linespacing=1.3 / 1.2)


def arrow(ax, x0, y0, x1, y1, style="-|>", dashed=False, colour="#444444"):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle=style, lw=0.8, color=colour, linestyle="--" if dashed else "-",
                                shrinkA=0, shrinkB=0, mutation_scale=8))


# F23: methodology
fig, ax = canvas(FULL_WIDTH, 6.0)
box(ax, 0.10, 5.25, 2.05, 0.60, "3,662 labelled fundus images, grades 0–4", title="APTOS 2019")
box(ax, 2.55, 5.25, 2.10, 0.60, "Exact duplicates removed: 158 images (123 groups; conflicting grades dropped)",
    title="Duplicate audit")
box(ax, 5.05, 5.25, 2.05, 0.60, "2,786 train / 357 validation / 361 test", title="Clean split")
arrow(ax, 2.15, 5.55, 2.55, 5.55); arrow(ax, 4.65, 5.55, 5.05, 5.55)
for xc, name, fill, edge in ((0.10, "Qwen3-VL-8B: general vision-language model", QWEN_FILL, QWEN_EDGE),
                             (3.70, "RetiZero: retina-specific foundation model", RZ_FILL, RZ_EDGE)):
    ax.add_patch(FancyBboxPatch((xc, 4.50), 3.40, 0.32, boxstyle="round,pad=0,rounding_size=0.06", facecolor=edge,
                                edgecolor=edge, linewidth=0.9))
    ax.text(xc + 1.70, 4.66, name, ha="center", va="center", fontsize=7.5, color="white", fontweight="bold")
    arrow(ax, xc + 1.70, 4.50, xc + 1.70, 4.30)
ax.plot([6.08, 6.08], [5.25, 5.05], color="#444444", lw=0.8)
ax.plot([1.80, 6.08], [5.05, 5.05], color="#444444", lw=0.8)
arrow(ax, 1.80, 5.05, 1.80, 4.82); arrow(ax, 5.40, 5.05, 5.40, 4.82)
lanes = [(0.10, QWEN_FILL, QWEN_EDGE,
          ["Crop to the field of view, pad to a square, resize to 512 px; CLAHE on the LAB lightness channel "
           "(an arm without CLAHE as control)",
           "QLoRA: 4-bit base model with a trainable LoRA adapter. Class-balanced sampling. Settings tested: "
           "learning rate, LoRA rank and alpha, CLAHE, adapter vs no adapter. Three seeds per setting",
           "Keep the epoch with the lowest validation loss (early stopping, patience 2)"],
          ["Preprocessing", "Fine-tuning", "Checkpoint"]),
         (3.70, RZ_FILL, RZ_EDGE,
          ["Crop to the field of view, pad to a square, resize to 224 px; no CLAHE (an arm with CLAHE as control)",
           "Full fine-tuning of the pretrained encoder and a new five-grade head. Class-weighted loss, training-only "
           "augmentation. Three seeds per setting",
           "Keep the epoch with the highest validation accuracy (early stopping, patience 4)"],
          ["Preprocessing", "Fine-tuning", "Checkpoint"])]
for x, fill, edge, texts, titles in lanes:
    ys, hs = [3.65, 2.45, 1.75], [0.65, 0.95, 0.50]
    for (yy, hh, t, ti) in zip(ys, hs, texts, titles):
        box(ax, x, yy, 3.40, hh, t, fill, edge, title=ti)
    arrow(ax, x + 1.70, 3.65, x + 1.70, 3.40); arrow(ax, x + 1.70, 2.45, x + 1.70, 2.25)
    arrow(ax, x + 1.70, 1.75, x + 1.70, 1.50)
box(ax, 0.10, 0.95, 7.00, 0.55,
    "Internal: APTOS-361 (clean test split). External: verified Messidor-2, 1,744 gradable images from 874 exams "
    "with adjudicated grades (never used for training or model choice)", title="Test sets")
box(ax, 0.10, 0.05, 3.40, 0.70, "Accuracy, quadratic weighted kappa, macro AUC, referable-DR sensitivity and "
    "specificity, calibration, error distance", title="Metrics")
box(ax, 3.70, 0.05, 3.40, 0.70, "Hierarchical bootstrap 95% CIs (images or exams × seeds); exact McNemar test "
    "per seed pair with Holm adjustment", title="Statistics")
arrow(ax, 1.80, 0.95, 1.80, 0.75); arrow(ax, 5.40, 0.95, 5.40, 0.75)
save(fig, "F23", "methodology", "Study workflow",
     "HEADLINE_VERIFICATION.md (split), run configurations (training settings)",
     "Study workflow. Exact duplicates were removed from APTOS 2019 before splitting. Each model was fine-tuned with "
     "its own preprocessing and training recipe, three seeds per setting, and its checkpoint was chosen on the "
     "validation set only. Both models were tested on the clean APTOS test split and on verified Messidor-2.")

# F24: Qwen3-VL-8B with the LoRA adapter and five-answer scoring
fig, ax = canvas(FULL_WIDTH, 3.55)
box(ax, 0.05, 2.25, 1.30, 0.95, "512 × 512 px, cropped, padded, CLAHE", QWEN_FILL, QWEN_EDGE, title="Fundus image")
box(ax, 0.05, 1.00, 1.30, 0.95, "Instruction and the five-grade scale", QWEN_FILL, QWEN_EDGE, title="Text prompt")
box(ax, 1.65, 2.10, 1.75, 1.25, "0.58 B parameters. 16 px patches: 1,024 patches; 2 × 2 merger: 256 visual tokens",
    SHARED_FILL, SHARED_EDGE, title="Vision encoder")
box(ax, 1.65, 1.00, 1.75, 0.80, "328 prompt tokens in all, 256 of them visual", SHARED_FILL, SHARED_EDGE,
    title="Tokenizer")
box(ax, 3.70, 1.00, 1.65, 2.35, "8.2 B parameters, 4-bit quantised, frozen. Reads the visual and text tokens "
    "and scores each candidate answer", SHARED_FILL, SHARED_EDGE, title="Language model")
box(ax, 5.65, 2.05, 1.50, 1.30, "Each of the five fixed answers <answer>k - LABEL</answer>, by total "
    "log-likelihood (5 passes)", QWEN_FILL, QWEN_EDGE, title="Answer scores")
box(ax, 5.65, 1.00, 1.50, 0.80, "Softmax over the five; grade = most likely", QWEN_FILL, QWEN_EDGE,
    title="Grade")
box(ax, 1.65, 0.05, 3.70, 0.65, "Rank 16, alpha 16, on the attention and MLP projections of both the vision encoder "
    "and the language model: 51.35 M trainable parameters (0.59%). Shown as the green tags above",
    TRAIN_FILL, TRAIN_EDGE, title="LoRA adapter")
for tx, ty in ((2.17, 2.16), (4.17, 1.06)):    # "+ LoRA" tags inside the vision encoder and the language model
    ax.add_patch(FancyBboxPatch((tx, ty), 0.71, 0.22, boxstyle="round,pad=0,rounding_size=0.04",
                                facecolor=TRAIN_FILL, edgecolor=TRAIN_EDGE, linewidth=0.8))
    ax.text(tx + 0.355, ty + 0.11, "+ LoRA", ha="center", va="center", fontsize=6.5, color="#006B4E",
            fontweight="bold")
arrow(ax, 1.35, 2.72, 1.65, 2.72); arrow(ax, 1.35, 1.40, 1.65, 1.40)
arrow(ax, 3.40, 2.72, 3.70, 2.72); arrow(ax, 3.40, 1.40, 3.70, 1.40)
arrow(ax, 5.35, 2.70, 5.65, 2.70); arrow(ax, 6.40, 2.05, 6.40, 1.80)
save(fig, "F24", "qwen_architecture", "Qwen3-VL-8B with the LoRA adapter",
     "parameter_counts.json and safetensors headers; scoring records (prompt_tokens, forward_passes_per_image)",
     "Qwen3-VL-8B as used here. The 4-bit base model is frozen; only the LoRA adapter (green) is trained, with "
     "next-token cross-entropy on the correct answer. At test time each of the five fixed answers is scored by its "
     "total log-likelihood, and the softmax over the five gives the grade probabilities.")

# F25: RetiZero with a new classification head
fig, ax = canvas(FULL_WIDTH, 2.55)
chain = [("Fundus image", "224 × 224 px, cropped, padded, no CLAHE", RZ_FILL, RZ_EDGE),
         ("Patch embedding", "16 px patches: 14 × 14 = 196", SHARED_FILL, SHARED_EDGE),
         ("ViT-L/16 encoder", "24 blocks, width 1,024, rank-8 LoRA branches; RetiZero pretrained weights",
          SHARED_FILL, SHARED_EDGE),
         ("Image feature", "1,024 values", SHARED_FILL, SHARED_EDGE),
         ("Linear head", "1,024 → 5, new, randomly initialised", RZ_FILL, RZ_EDGE),
         ("Grade", "Softmax over the five grades; grade = most likely", RZ_FILL, RZ_EDGE)]
w, gap, x = 1.02, 0.188, 0.05
for title, text, fill, edge in chain:
    box(ax, x, 1.05, w, 1.40, text, fill, edge, title=title)
    if x + w + gap < FULL_WIDTH:
        arrow(ax, x + w, 1.75, x + w + gap, 1.75)
    x += w + gap
box(ax, 0.05, 0.05, 7.10, 0.75, "All 304.1 M weights are trained (encoder, LoRA branches and head). Cross-entropy "
    "weighted by inverse class frequency; AdamW, learning rate 7.68e-5 from a 25-trial search on validation accuracy, "
    "cosine schedule; random crop, flip and rotation during training only", TRAIN_FILL, TRAIN_EDGE, title="Training")
save(fig, "F25", "retizero_architecture", "RetiZero with a new classification head",
     "train_retizero.py configuration; parameter_counts.json; legacy/original_learning_rate_search.csv",
     "RetiZero as used here: the pretrained RetiZero image encoder (ViT-L/16 with rank-8 LoRA branches) and a new "
     "linear head. Unlike Qwen, every weight is trained.")

## Figure index and layout report

In [ ]:
index = pd.DataFrame(INDEX)
index.to_csv(OUT / "figure_index.csv", index=False)
print(f"{len(index)} figures written to {OUT}")
if LAYOUT_PROBLEMS:
    print("\nFigures with overlapping text:")
    for fid, hits in LAYOUT_PROBLEMS.items():
        print(f"  {fid}: " + "; ".join(hits))
else:
    print("Layout check: no overlapping text in any figure.")
REPORT += [f"- {len(index)} figures written; layout check: "
           + ("no overlapping text in any figure" if not LAYOUT_PROBLEMS else f"overlaps in {', '.join(LAYOUT_PROBLEMS)}"),
           f"- Finished {datetime.now():%Y-%m-%d %H:%M:%S}", ""]
REPORT += [f"  - {fid}: " + "; ".join(h) for fid, h in LAYOUT_PROBLEMS.items()]
plain = [line.replace("`", "").removeprefix("# ") for line in REPORT]  # plain text, not Markdown
(OUT / "run_report.txt").write_text("\n".join(plain) + "\n", encoding="utf-8")
print("Run report:", OUT / "run_report.txt")

## Package everything for the report writer

Zips everything listed in `docs/paper/HANDOFF_REPORT_WRITER.md`:

- this folder of figures
- the analysis tables
- the evidence and checklist documents

The zip goes in `D:\Afterimage_data_share\`. It holds metrics, image IDs and plots only: no fundus images and no model weights.

Skipped when the notebook runs inside the results package: the package already holds all of this.

In [ ]:
import zipfile
from datetime import date

def make_bundle():
    bundle = Path(r"D:\Afterimage_data_share") / f"paper_report_bundle_{date.today():%Y%m%d}.zip"
    members = {
        "figures": sorted(OUT.glob("*")),
        "tables": sorted(p for p in TABLES.rglob("*") if p.is_file()),
        "docs": [REPO / "docs/paper/HANDOFF_PROGRESS_FOR_CHATGPT.md", REPO / "docs/paper/HANDOFF_FIGURES_GUIDE.md",
                 REPO / "docs/paper/HANDOFF_REPORT_WRITER.md", REPO / "docs/paper/REVISION_EVIDENCE.md",
                 REPO / ".claude/handoffs/2026-11-15-182947-afterimage-paper-progress.md",
                 REPO / "REVIEWER_CHECKLIST.md", REPO / "docs/paper/REMAINING_WORK.md", REPO / "docs/paper/PLAN.md",
                 REPO / "docs/paper/DELIVERABLES_CHECKLIST.md", REPO / "docs/paper/HEADLINE_VERIFICATION.md",
                 REPO / "docs/paper/APTOS_DUPLICATES_LITERATURE.md", REPO / "finetuning/paper_tables/README.md"],
    }
    missing = [str(p) for paths in members.values() for p in paths if not p.is_file()]
    assert not missing, missing
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
        for folder, paths in members.items():
            for path in paths:
                arcname = f"{folder}/{path.relative_to(TABLES).as_posix()}" if folder == "tables" else f"{folder}/{path.name}"
                zf.write(path, arcname)
    print(f"{sum(len(v) for v in members.values())} files, {bundle.stat().st_size / 2**20:.1f} MiB -> {bundle}")


if PACKAGE:
    print("Inside the results package: nothing to bundle; the package already holds the figures, tables and documents.")
else:
    make_bundle()